# FashionCLIP + SCHP + DINOv2 Retrieval
1. **SCHP** (SegFormer on ATR) — pixel-level clothing segmentation
2. **FashionCLIP** — semantic label for each region
3. **DINOv2 + catalog** — exact product retrieval for shoes

Run cells 1–4 once per session. Re-run cells 5–10 per image.

In [ ]:
from PIL import Image
import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from transformers import (
    AutoProcessor,
    AutoModelForZeroShotImageClassification,
    SegformerImageProcessor,
    AutoModelForSemanticSegmentation,
    AutoImageProcessor,
    AutoModel,
)

In [ ]:
# ── Load FashionCLIP [run once] ───────────────────────────────────────────────
device = 'cuda' if torch.cuda.is_available() else 'cpu'

clip_processor = AutoProcessor.from_pretrained('patrickjohncyh/fashion-clip')
clip_model = AutoModelForZeroShotImageClassification.from_pretrained(
    'patrickjohncyh/fashion-clip'
).to(device)
clip_model.eval()
print(f'FashionCLIP loaded on {device}')

In [ ]:
# ── Load SCHP parser + DINOv2 [run once] ─────────────────────────────────────
ATR_LABELS = [
    'Background', 'Hat', 'Hair', 'Sunglasses', 'Upper-clothes',
    'Skirt', 'Pants', 'Dress', 'Belt', 'Left-shoe', 'Right-shoe',
    'Face', 'Left-leg', 'Right-leg', 'Left-arm', 'Right-arm', 'Bag', 'Scarf'
]
APPAREL_IDX = {1, 4, 5, 6, 7, 8, 9, 10, 16, 17}

schp_processor = SegformerImageProcessor.from_pretrained('mattmdjaga/segformer_b2_clothes')
schp_model = AutoModelForSemanticSegmentation.from_pretrained(
    'mattmdjaga/segformer_b2_clothes'
).to(device)
schp_model.eval()
print(f'SCHP loaded on {device}')

dino_processor = AutoImageProcessor.from_pretrained('facebook/dinov2-base')
dino_model = AutoModel.from_pretrained('facebook/dinov2-base').to(device)
dino_model.eval()
print(f'DINOv2 loaded on {device}')

In [ ]:
# ── Catalog definition — edit freely ─────────────────────────────────────────
# name → DDG image search query used to auto-download the product image
CATALOG = {
    'Adidas Samba OG White Black':         'adidas samba og white black sneaker product',
    'Adidas Samba OG Black White Gum':     'adidas samba og black white gum sole sneaker',
    'ASICS GT-2160 Cream Pure Silver':     'asics gt-2160 cream pure silver sneaker product',
    'ASICS GT-2160 White Grove':           'asics gt-2160 white grove sneaker product',
    'ASICS Gel-Kayano 14 Cream':           'asics gel-kayano 14 cream rose gold sneaker',
    'New Balance 530 White Silver':        'new balance 530 white silver sneaker product',
    'New Balance 550 White Green':         'new balance 550 white green sneaker product',
    'New Balance 9060 Grey':               'new balance 9060 grey sneaker product',
    'Nike Dunk Low White Black Panda':     'nike dunk low white black panda sneaker',
    'Nike Air Force 1 White':              'nike air force 1 white sneaker product',
    'Nike P-6000 Metallic Silver':         'nike p-6000 metallic silver sneaker product',
    'Jordan 1 Retro High Chicago':         'air jordan 1 retro high og chicago sneaker',
    'Converse Chuck Taylor All Star White':'converse chuck taylor all star white low sneaker',
    'Vans Old Skool Black White':          'vans old skool black white sneaker product',
    'Onitsuka Tiger Mexico 66 White Red':  'onitsuka tiger mexico 66 white red sneaker product',
}

# FashionCLIP labels per SCHP class
SCHP_TO_LABELS = {
    'Hat':          ['hat', 'cap', 'beanie', 'bucket hat'],
    'Sunglasses':   ['sunglasses', 'glasses'],
    'Upper-clothes':['t-shirt', 'shirt', 'button-up shirt', 'hoodie', 'sweatshirt',
                     'jacket', 'coat', 'jersey', 'tank top', 'nike jersey'],
    'Skirt':        ['skirt', 'mini skirt', 'midi skirt', 'maxi skirt'],
    'Pants':        ['blue jeans', 'pants', 'trousers', 'shorts', 'sweatpants'],
    'Dress':        ['dress', 'maxi dress', 'mini dress', 'sundress'],
    'Belt':         ['belt', 'leather belt'],
    'Bag':          ['bag', 'backpack', 'sling bag', 'tote bag', 'handbag'],
    'Scarf':        ['scarf', 'neck scarf'],
}

# Shoe segments use catalog retrieval instead of FashionCLIP
SHOE_LABELS = {'Left-shoe', 'Right-shoe'}

MIN_MASK_PX    = 500
MIN_CLIP_SCORE = 0.2
TOP_K          = 3    # top catalog matches to show per shoe crop

In [ ]:
# ── Scan catalog directory ────────────────────────────────────────────────────
# Drop product images into ./catalog/ with any filename.
# Name becomes the display label (underscores → spaces).
CATALOG_DIR = Path('./catalog')
CATALOG_DIR.mkdir(exist_ok=True)

catalog_files = sorted(
    p for p in CATALOG_DIR.iterdir()
    if p.suffix.lower() in {'.jpg', '.jpeg', '.png', '.webp'}
)

print(f"Found {len(catalog_files)} catalog images:")
for p in catalog_files:
    print(f"  {p.name}")

In [ ]:
# ── Build DINOv2 embedding index ─────────────────────────────────────────────
def dino_embed(img_pil):
    inp = dino_processor(images=img_pil, return_tensors='pt').to(device)
    with torch.no_grad():
        feat = dino_model(**inp).last_hidden_state[:, 0]  # CLS token
    return F.normalize(feat, dim=-1).cpu().numpy()[0]

catalog_names, catalog_embeddings, catalog_paths = [], [], []

for path in catalog_files:
    try:
        img = Image.open(path).convert('RGB')
        label = path.stem.replace('_', ' ')
        catalog_names.append(label)
        catalog_embeddings.append(dino_embed(img))
        catalog_paths.append(path)
        print(f"  ✓ {label}")
    except Exception as e:
        print(f"  ✗ {path.name}: {e}")

catalog_embeddings = np.stack(catalog_embeddings)
print(f"\nIndex ready: {len(catalog_names)} products")

In [ ]:
# ── Load image + run SCHP parsing ─────────────────────────────────────────────
IMAGE_PATH = '/Users/hanavmodasiya/Downloads/2c2b22425837ede3a3b0c61833f5a947.jpg'

image = Image.open(IMAGE_PATH).convert('RGB')
image_np = np.array(image)
h, w = image_np.shape[:2]

inp = schp_processor(images=image, return_tensors='pt').to(device)
with torch.no_grad():
    logits = schp_model(**inp).logits

parsing = (
    F.interpolate(logits, size=(h, w), mode='bilinear', align_corners=False)
    .squeeze(0).argmax(0).cpu().numpy()
)

found = [ATR_LABELS[i] for i in sorted(np.unique(parsing)) if i in APPAREL_IDX]
print(f'Parsing done. Apparel regions: {found}')

In [ ]:
# ── Classify: FashionCLIP for apparel, DINOv2 retrieval for shoes ─────────────
def retrieve(crop_pil, k=TOP_K):
    q  = dino_embed(crop_pil)
    qf = dino_embed(crop_pil.transpose(Image.FLIP_LEFT_RIGHT))
    # take element-wise max similarity across original and flipped
    sims = np.maximum(catalog_embeddings @ q, catalog_embeddings @ qf)
    idx = np.argsort(sims)[::-1][:k]
    return [(catalog_names[i], float(sims[i]), catalog_paths[i]) for i in idx]

results = []

for class_idx in sorted(np.unique(parsing)):
    if class_idx not in APPAREL_IDX:
        continue

    schp_label = ATR_LABELS[class_idx]
    mask = (parsing == class_idx)
    if mask.sum() < MIN_MASK_PX:
        continue

    ys, xs = np.where(mask)
    crop = image.crop((xs.min(), ys.min(), xs.max(), ys.max()))

    if schp_label in SHOE_LABELS:
        matches = retrieve(crop)
        fine_label = matches[0][0]
        score = matches[0][1]
    else:
        labels = SCHP_TO_LABELS.get(schp_label, [schp_label])
        inp = clip_processor(images=crop, text=labels, return_tensors='pt', padding=True).to(device)
        with torch.no_grad():
            probs = clip_model(**inp).logits_per_image.softmax(dim=-1)[0]
        top_idx = probs.argmax().item()
        score = probs[top_idx].item()
        fine_label = labels[top_idx] if score >= MIN_CLIP_SCORE else schp_label
        matches = None

    results.append({
        'schp_label': schp_label,
        'fine_label': fine_label,
        'score': round(score, 3),
        'mask': mask,
        'crop': crop,
        'matches': matches,
    })
    tag = '(retrieval)' if schp_label in SHOE_LABELS else '(clip)'
    print(f'{schp_label:15s} → {fine_label:35s} {score:.3f}  {tag}')

In [ ]:
# ── Visualize all segments + retrieval matches ────────────────────────────────
for idx, r in enumerate(results):
    is_shoe = r['schp_label'] in SHOE_LABELS
    ncols = TOP_K + 2 if is_shoe else 1
    fig, axes = plt.subplots(1, ncols, figsize=(5 * ncols, 5))

    # Left panel: segment overlay on full image
    color = plt.colormaps['tab20'](idx % 20)[:3]
    overlay = np.zeros((*r['mask'].shape, 4), dtype=float)
    overlay[r['mask']] = [*color, 0.5]
    axes[0].imshow(image_np)
    axes[0].imshow(overlay)
    axes[0].set_title(f"{r['schp_label']}\n→ {r['fine_label']}  ({r['score']:.2f})",
                      fontsize=9, fontweight='bold')
    axes[0].axis('off')

    # Shoe panels: crop + top-K catalog matches
    if is_shoe:
        axes[1].imshow(r['crop'])
        axes[1].set_title('crop', fontsize=8)
        axes[1].axis('off')
        for i, (name, score, path) in enumerate(r['matches']):
            axes[i + 2].imshow(Image.open(path).convert('RGB'))
            axes[i + 2].set_title(f"{name}\n{score:.3f}", fontsize=8)
            axes[i + 2].axis('off')

    plt.tight_layout()
    plt.savefig(f"output_{r['schp_label'].replace(' ', '_')}.png", dpi=150, bbox_inches='tight')
    plt.show()